# Example 1.2.4 / 1: Values Only on Listed Pairs

Companion notebook for **1.2.4 Sparse Parameters** in the AMPL lab Documentation, section "Values Only on Listed Pairs". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Values Only on Listed Pairs

Suppose only four shipping routes exist:


`GARY` to `FRA` and `PITT` to `LAN` are not routes at all. Indexing `cost` over `LINKS` creates
four parameter values, not all six combinations in `ORIG cross DEST`. In `data.py` and `run.py`, `ampl` is an
`AMPL` object that has already read `model.mod`. The pair set is assigned a list of Python
tuples, the parameter is assigned a dictionary with tuple keys, and `to_dict()` returns those
stored values:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIG;
set DEST;
set LINKS within {ORIG, DEST};   # routes that exist
param cost {LINKS} >= 0;         # one cost per existing route

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIG"] = ["GARY", "PITT"]
ampl.set["DEST"] = ["FRA", "DET", "LAN"]
ampl.set["LINKS"] = [("GARY", "DET"), ("GARY", "LAN"),
                     ("PITT", "FRA"), ("PITT", "DET")]
ampl.param["cost"] = {("GARY", "DET"): 14, ("GARY", "LAN"): 11,
                      ("PITT", "FRA"): 27, ("PITT", "DET"): 16}

In [5]:
# run.py
print(ampl.param["cost"].to_dict())
print(ampl.param["cost"]["PITT", "FRA"])

{('GARY', 'DET'): 14, ('GARY', 'LAN'): 11, ('PITT', 'DET'): 16, ('PITT', 'FRA'): 27}
27


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ORIG := GARY PITT;
set DEST := FRA DET LAN;
set LINKS := (GARY,DET) (GARY,LAN) (PITT,FRA) (PITT,DET);
param cost :=
  GARY DET  14
  GARY LAN  11
  PITT FRA  27
  PITT DET  16 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display cost;

cost :=
GARY DET   14
GARY LAN   11
PITT DET   16
PITT FRA   27
;

